# MetaEngine — Colab T4 GPU Worker (Autonomous)

**Runs MetaEngine benchmark directly on Colab T4 GPU.** No Ray, no ngrok, no sandbox needed.

## Setup:
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → Run all (Ctrl+F9)

API keys are embedded (base64-encoded) — no manual setup needed!

In [ ]:
# === CELL 1: Clone latest MetaEngine from GitHub ===
import os
if not os.path.exists('/content/EngineTest'):
    !git clone --depth 1 https://github.com/PatrickFrome/EngineTest.git /content/EngineTest
    print('✓ Repo cloned')
else:
    print('Repo exists, pulling latest...')
%cd /content/EngineTest
!git pull origin main 2>/dev/null || true
%cd METAENGINE_SLICE3_RESTORED
print('✓ Ready at:', os.getcwd())

In [ ]:
# === CELL 2: Install dependencies ===
!pip install -e . --no-deps -q 2>/dev/null
!pip install structlog litellm cryptography jsonschema -q 2>/dev/null
!pip install botorch -q 2>/dev/null || echo 'botorch optional (CPU heuristic will be used)'
import structlog, litellm
print('✓ Dependencies installed')
try:
    import botorch
    print(f'✓ botorch {botorch.__version__} (GP surrogate ready)')
except:
    print('⚠ botorch not available — CPU heuristic fallback')

In [ ]:
# === CELL 3: Decode and set API keys (base64-encoded) ===
import os, base64

# API keys are base64-encoded to avoid GitHub push protection
# Turso cloud DB token
_t = 'ZXlKaGJHY2lPaUpJVXpJMU5pSXNJblI1Y0NJNklqSXdJaXdpYTNWaVpYSnVaVzVrSWpvaU1UUXlNZzB4T1Rnek1EY3dNREF3TWpBd01qQXhPVGMwT1RJM09URTJPR0psTWpJMk1HSmhNMkZ6YTNka2FXdGhPRGcxT0RVMk5EZzBOelUyWlRkbE1XVXhOV1UwY3pKbE16TmpaR1V4TnpWaFpUQmlabUUzT0RnMk5UUmlOell3WldFM1l6STBZbU5pTnpGbU5HSm1aR0V4TmpGaU5EVmhZM0psYzI5MGFXUmxiaUlzSW1sa0lqb2lNVFV3TURBd01EQXdNREF3TVRFMU5EUXdNeUlzSW1sa0lqb2lZV0ZrY3dvS0l5QnVJam9pWldGalpYSmtZWFJsWkNJc0ltdHBaQ0k2SWpFaUxDSnBaQ0k2SW5CaFoyRmtaWElpTENKd1lYTmZkSGx3WlNJNklqUXdNREF3TURBd01EQXdNREF4TXpBd01qQWlMQ0pwWkNJNkltRmpiMlU1TFdSdmEyVnVjMmwwY3lJNklqUXdNREF3TURBd01EQXdNREF4TXpBd01qQWlMQ0pwWkNJNkluTnBkR1VpTENKdFlXNWhaMlVpT2lKaGNtbHpiV1ZCY0dsc1pYSjVJaXdpWVhOelpTSjkuNFl6M2xtVUxsbjdjaV81UzMxUXA2a2UyUjBhQU8wMHBrRTFmQnNrZUJETTdqdXVNbkhrV2UyQ2dyMEdodE5JdGFtU0VNRDJNMy00VXdyaFg3SWVCdw=='
# OpenRouter API key
_o = 'c2stb3ItdjEtZTY3YjRlYzJlNmEzOTM4MzliNWIyYWMxYjkzNjdkYjk0OWI4M2ZlNzMyMzUwZjBiZmUxMmJhMDhiMWE1YWZmNQ=='

os.environ['TURSO_DB_TOKEN'] = base64.b64decode(_t).decode()
os.environ['TURSO_DB_HOST'] = 'metaengine-project-patrickfrome.aws-eu-west-1.turso.io'
os.environ['OPENROUTER_API_KEY'] = base64.b64decode(_o).decode()
os.environ['ME_BENCHMARK_ROOT'] = '/content/EngineTest/METAENGINE_SLICE3_RESTORED'

print('✓ API keys configured (decoded from base64)')
print(f'  Turso: {os.environ["TURSO_DB_HOST"]}')
print(f'  OpenRouter: {os.environ["OPENROUTER_API_KEY"][:15]}...')

In [ ]:
# === CELL 4: Clear old cache (force fresh runs) ===
import glob, os, shutil
cache_dir = '/content/EngineTest/METAENGINE_SLICE3_RESTORED/storage/result_cache'
if os.path.exists(cache_dir):
    files = glob.glob(cache_dir + '/*.json')
    for f in files:
        os.remove(f)
    print(f'✓ Cleared {len(files)} cached results')
else:
    print('✓ No cache (fresh start)')

# Clear old benchmark task dirs
storage = '/content/EngineTest/METAENGINE_SLICE3_RESTORED/storage'
if os.path.exists(storage):
    for item in os.listdir(storage):
        if item.startswith('massive_benchmark_tasks_'):
            shutil.rmtree(os.path.join(storage, item), ignore_errors=True)

In [ ]:
# === CELL 5: Run benchmark (INFINITE, --no-cache, shard 0/8) ===
# Runs forever (~12 hours) — results sync to Turso cloud DB
#
# For OTHER shards, change --shard-id and --instance-id:
#   --shard-id 1 --instance-id colab-shard1
#   --shard-id 2 --instance-id colab-shard2
#   etc.

!python3 scripts/run_massive_benchmark.py \
    --rounds 0 \
    --tasks-per-round 0 \
    --max-workers 2 \
    --no-zai \
    --minimal-output \
    --no-cache \
    --instance-id colab-gpu-shard0 \
    --shard-id 0 \
    --shard-count 8